# Antelligence PPO Training (Colab GPU)

Runs `train_ppo.py` on a Colab GPU runtime and saves weights/artifacts to Google Drive.

**Usage:** Connect via VS Code Colab extension or run directly in Colab. Ensure runtime is set to GPU.

In [4]:
# Parameters
REPO_URL = 'https://github.com/eren23/antelligence.git'
BRANCH = 'imp/learning-1'
BRAIN = 'torch_nn'           # 'torch_nn' or 'torch_transformer'
TICKS = 50_000
ANTS = 200
LR = 3e-4
ROLLOUT_LENGTH = 1024
SAVE_INTERVAL = 10_000
SEED = 42
LOAD_WEIGHTS = None           # e.g. '/content/drive/MyDrive/attocode_runs/weights/best'
DRIVE_OUT_ROOT = '/content/drive/MyDrive/attocode_runs'
WORKDIR = '/content/work'

In [5]:
# GPU Check
import torch
assert torch.cuda.is_available(), "No GPU — change Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
!nvidia-smi

GPU: NVIDIA L4
VRAM: 23.7 GB
Tue Mar 10 13:41:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             15W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os
import subprocess

repo_dir = os.path.join(WORKDIR, 'repo')
os.makedirs(WORKDIR, exist_ok=True)

if not os.path.isdir(os.path.join(repo_dir, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, repo_dir], check=True)

subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=repo_dir, check=True)
subprocess.run(['git', 'checkout', BRANCH], cwd=repo_dir, check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=repo_dir, check=True)
print('Repo ready:', repo_dir)

Repo ready: /content/work/repo


In [8]:
!pip install -q pygame numpy pyyaml torch

In [9]:
# Live GPU Monitoring (runs in background during training)
import subprocess, threading, time

gpu_log = []  # list of (timestamp, gpu_util%, mem_used_mb)
_stop_monitor = threading.Event()

def _monitor_gpu(interval=10):
    while not _stop_monitor.is_set():
        try:
            out = subprocess.check_output(
                ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used',
                 '--format=csv,noheader,nounits'], text=True)
            gpu_pct, mem_mb = [float(x) for x in out.strip().split(',')]
            gpu_log.append((time.time(), gpu_pct, mem_mb))
        except Exception:
            pass
        _stop_monitor.wait(interval)

threading.Thread(target=_monitor_gpu, daemon=True).start()
print('GPU monitor started')

GPU monitor started


In [10]:
# Run PPO Training
import subprocess, re, os, sys
from datetime import datetime

ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
out_dir = os.path.join(DRIVE_OUT_ROOT, f'ppo_{ts}')
os.makedirs(out_dir, exist_ok=True)

cmd = [
    sys.executable, 'train_ppo.py',
    '--brain', BRAIN,
    '--ticks', str(TICKS),
    '--ants', str(ANTS),
    '--lr', str(LR),
    '--rollout-length', str(ROLLOUT_LENGTH),
    '--save-interval', str(SAVE_INTERVAL),
    '--seed', str(SEED),
    '--output-dir', out_dir,
    '--config', 'colony_config.yaml',
]
if LOAD_WEIGHTS:
    cmd += ['--load-weights', LOAD_WEIGHTS]

print('Running:', ' '.join(cmd))

# Parse progress lines: "[torch_nn] Tick 5,000 | Pop 200 | Food 1234.5 | Deaths 12"
train_log = []  # list of dicts: {tick, pop, food, deaths}
_progress_re = re.compile(
    r'Tick\s+([\d,]+)\s+\|\s+Pop\s+(\d+)\s+\|\s+Food\s+([\d.]+)\s+\|\s+Deaths\s+(\d+)'
)

proc = subprocess.Popen(
    cmd, cwd=repo_dir,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end='')  # real-time output in notebook
    m = _progress_re.search(line)
    if m:
        train_log.append({
            'tick': int(m.group(1).replace(',', '')),
            'pop': int(m.group(2)),
            'food': float(m.group(3)),
            'deaths': int(m.group(4)),
        })

proc.wait()
_stop_monitor.set()  # stop GPU monitor thread
print(f'\nExit code: {proc.returncode}')
print(f'Artifacts: {out_dir}')

/tmp/ipykernel_9526/1516683429.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')


Running: /usr/bin/python3 train_ppo.py --brain torch_nn --ticks 50000 --ants 200 --lr 0.0003 --rollout-length 1024 --save-interval 10000 --seed 42 --output-dir /content/drive/MyDrive/attocode_runs/ppo_20260310_134224 --config colony_config.yaml


KeyboardInterrupt: 

In [ ]:
# Training Summary
import json

summary_path = os.path.join(out_dir, 'ppo_summary.json')
try:
    with open(summary_path) as f:
        summary = json.load(f)
    print(json.dumps(summary, indent=2))
except FileNotFoundError:
    print(f'Summary not found at {summary_path}')
    print('Training may not have completed successfully.')

In [ ]:
# Training Curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Subplot 1: Food collected over ticks
if train_log:
    ticks = [d['tick'] for d in train_log]
    food = [d['food'] for d in train_log]
    axes[0].plot(ticks, food, linewidth=1)
    axes[0].set_xlabel('Tick')
    axes[0].set_ylabel('Food Stored')
    axes[0].set_title('Food Collected')
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No training data', ha='center', va='center',
                 transform=axes[0].transAxes)

# Subplot 2: GPU utilization over time
if gpu_log:
    t0 = gpu_log[0][0]
    elapsed = [(t - t0) / 60 for t, _, _ in gpu_log]
    util = [u for _, u, _ in gpu_log]
    axes[1].plot(elapsed, util, linewidth=1, color='tab:orange')
    axes[1].set_xlabel('Time (min)')
    axes[1].set_ylabel('GPU Utilization %')
    axes[1].set_title('GPU Utilization')
    axes[1].set_ylim(0, 100)
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No GPU data', ha='center', va='center',
                 transform=axes[1].transAxes)

# Subplot 3: GPU memory over time
if gpu_log:
    mem = [m for _, _, m in gpu_log]
    axes[2].plot(elapsed, mem, linewidth=1, color='tab:green')
    axes[2].set_xlabel('Time (min)')
    axes[2].set_ylabel('Memory Used (MB)')
    axes[2].set_title('GPU Memory')
    axes[2].grid(True, alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'No GPU data', ha='center', va='center',
                 transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# Inspect Saved Weights
import numpy as np
from pathlib import Path

weight_dir = Path(out_dir)
npz_files = sorted(weight_dir.glob('*.npz'))
if npz_files:
    for f in npz_files:
        data = np.load(f)
        total = sum(v.size for v in data.values())
        print(f"{f.name}: {len(data)} arrays, {total:,} params")
else:
    print('No .npz weight files found in', out_dir)

In [ ]:
# Copy Weights to Drive (if output wasn't already on Drive)
import shutil

drive_backup = os.path.join(DRIVE_OUT_ROOT, 'weights_backup', f'ppo_{ts}')
if not out_dir.startswith('/content/drive'):
    os.makedirs(drive_backup, exist_ok=True)
    for f in Path(out_dir).iterdir():
        shutil.copy2(str(f), drive_backup)
    print(f'Copied to: {drive_backup}')
else:
    print(f'Artifacts already on Drive: {out_dir}')

## Overnight Training Alternative

For longer runs with graceful Ctrl+C support and best-checkpoint tracking, use `train_overnight.py`.

In [ ]:
# Overnight training (optional — uncomment to run)
# !cd {repo_dir} && python3 train_overnight.py \
#     --brain {BRAIN} --ticks 100000 --ants {ANTS} --seed {SEED} \
#     --output-dir {out_dir} --save-every 5000